# 🚉 RailSetu M3 — Rail Defect Detection

Run photos of railway track through two trained models.

| Model | Question it answers | Output |
|---|---|---|
| **A** — EfficientNet-B0 | *What kind of damage?* | `flaking` / `shelling` / `spalling` / `squat` + confidence |
| **B** — YOLO11-s | *Where is the damage?* | boxes around defects |

**You do not need to train anything.** The models are already trained; this notebook
just downloads them and runs your photos through.

**Runtime > Run all**, then scroll down. Takes about a minute.

---
### ⚠️ One honest limitation
Model B learned from *tight close-ups of the rail only*. On wide photos that include
ballast (gravel), it boxes the gravel — it has never seen gravel and mistakes the
texture for damage. Trust Model B on `railhead_crops/`, not `rail_frames/`.

## 1 · Setup

In [ ]:
!pip install -q ultralytics
print("ready")

## 2 · Download the models

Paste the Google Drive share link for `railsetu-m3-demo.zip` below.
In Drive: right-click the file → **Share** → *Anyone with the link* → **Copy link**.

In [ ]:
DRIVE_LINK = ""  # <-- paste your Google Drive link between the quotes

import os, re, zipfile

if DRIVE_LINK.strip():
    m = re.search(r"/d/([A-Za-z0-9_-]+)|id=([A-Za-z0-9_-]+)", DRIVE_LINK)
    if not m:
        raise SystemExit("Could not read a file id from that link.")
    file_id = m.group(1) or m.group(2)
    !pip install -q gdown
    !gdown -q --id {file_id} -O bundle.zip
    with zipfile.ZipFile("bundle.zip") as z:
        z.extractall("m3")
    print("downloaded and unpacked")
else:
    from google.colab import files
    print("No link set - upload railsetu-m3-demo.zip manually:")
    up = files.upload()
    with zipfile.ZipFile(list(up)[0]) as z:
        z.extractall("m3")

print(sorted(os.listdir("m3")))

## 3 · Load both models

In [ ]:
import torch, glob
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_b0
from ultralytics import YOLO

ROOT = Path("m3")
CLASSES = ["flaking", "shelling", "spalling", "squat"]
device = "cuda" if torch.cuda.is_available() else "cpu"

ck = torch.load(ROOT/"model_a_best.pt", map_location="cpu", weights_only=False)
model_a = efficientnet_b0(weights=None)
model_a.classifier[1] = torch.nn.Linear(model_a.classifier[1].in_features, len(ck["classes"]))
model_a.load_state_dict(ck["state_dict"]); model_a.to(device).eval()

_b = ROOT/"model_b_v2_best.pt"
if not _b.exists(): _b = ROOT/"model_b_best.pt"   # older bundles
model_b = YOLO(str(_b))
print("localizer:", _b.name)

tf = transforms.Compose([
    transforms.Resize(257), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

print(f"loaded both models on {device} | classes: {ck['classes']}")

## 4 · The analyse function

In [ ]:
import matplotlib.pyplot as plt
from PIL import ImageDraw

def analyse(path, run_model_b=True, conf=0.25, show=True):
    img = Image.open(path).convert("RGB")

    with torch.no_grad():
        probs = torch.softmax(model_a(tf(img).unsqueeze(0).to(device)), 1)[0].cpu().numpy()
    top = int(probs.argmax())
    label, confidence = CLASSES[top], float(probs[top])

    boxes = []
    if run_model_b:
        for b in model_b.predict(source=str(path), conf=conf, verbose=False)[0].boxes:
            boxes.append((b.xyxy[0].tolist(), float(b.conf[0])))

    if show:
        canvas = img.copy(); d = ImageDraw.Draw(canvas)
        for (x0,y0,x1,y1), s in boxes:
            d.rectangle([x0,y0,x1,y1], outline=(0,255,90), width=4)
            d.text((x0+5, max(0,y0-12)), f"defect {s:.2f}", fill=(0,255,90))
        plt.figure(figsize=(9,5)); plt.imshow(canvas); plt.axis("off")
        flag = "  (low confidence)" if confidence < 0.5 else ""
        plt.title(f"{label} - {confidence:.0%} sure{flag}   |   {len(boxes)} box(es)", fontsize=13)
        plt.show()
        print("   " + "  ".join(f"{c}:{p:.0%}" for c,p in zip(CLASSES, probs)))

    return {"label": label, "confidence": round(confidence,3),
            "probabilities": {c: round(float(p),3) for c,p in zip(CLASSES,probs)},
            "boxes": boxes}

print("ready")

## 5 · Model A on real track photos

These 28 photos were **held out** — Model A never saw them during training.
The filename tells you the correct answer, so you can mark it yourself.

Model B is switched off here (this is the wide-photo case where it boxes gravel).

In [ ]:
for p in sorted(glob.glob("m3/rail_frames/*.jpg"))[:6]:
    truth = Path(p).stem.split("_")[0]
    r = analyse(p, run_model_b=False)
    print(f"   truth: {truth}   ->   predicted: {r['label']}   "
          f"{'CORRECT' if truth == r['label'] else 'WRONG'}\n")

## 6 · Model B on railhead close-ups

This is Model B's home ground — tight crops of the rail with no ballast in frame.

In [ ]:
for p in sorted(glob.glob("m3/railhead_crops/*.jpg"))[:6]:
    analyse(p, run_model_b=True)
    print()

## 7 · Score Model A yourself

Runs all 28 held-out photos and counts how many it got right.

In [ ]:
from collections import Counter
hits, per_class = 0, Counter(); totals = Counter()
paths = sorted(glob.glob("m3/rail_frames/*.jpg"))
for p in paths:
    truth = Path(p).stem.split("_")[0]
    got = analyse(p, run_model_b=False, show=False)["label"]
    totals[truth] += 1
    if got == truth: hits += 1; per_class[truth] += 1

print(f"Overall: {hits}/{len(paths)} = {hits/len(paths):.0%}\n")
for c in CLASSES:
    if totals[c]:
        print(f"  {c:9} {per_class[c]}/{totals[c]}")
print("\nExpected: strong on flaking and squat, weak on spalling and shelling —")
print("those two had only ~20 real defects in the training data.")

## 8 · Try your own photo

Run this cell and upload any photo of railway track.

In [ ]:
from google.colab import files
up = files.upload()
for name in up:
    analyse(name, run_model_b=True)

---
### Data attribution — both CC BY 4.0
* Arain et al., *Railway Track Surface Faults Dataset*, Mendeley Data, doi:10.17632/8hxtgyyxrw.2
* Niu et al., *RSDDS*, IEEE TII 2021 / IEEE TIM 2021